# 🖥️ CUDA 베이스라인 (Colab/Runpod) — RT-DETR · Cascade R-CNN · DINO

MPS로 학습 불가한 모델을 **동일 fold0 번들**로 CUDA에서 돌린다. MPS 5종과 **같은 mAP@[0.75:0.95] 하니스**를 써서 결과를 그대로 합류.

## 실행 방법
1. **런타임 → GPU** 로 변경(Colab: 런타임 유형 변경 → T4/GPU).
2. `lhk_cuda_bundle.zip` 을 업로드(왼쪽 파일창 드래그) — 또는 Google Drive에 두고 마운트.
3. 위에서부터 셀 실행. **RT-DETR 먼저**(가장 안정) → mmdet(Cascade/DINO)는 설치 버전 이슈가 있을 수 있어 best-effort.
4. 완료 후 `cuda_baselines.json` 을 다운로드해 KAI에게 전달 → MPS 표에 합류.

## 0. 셋업 · 번들 언집 · 공통 하니스

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import os, zipfile
from pathlib import Path
import torch
BUNDLE_ZIP = "/content/drive/MyDrive/lhk_cuda_bundle.zip"        # 업로드한 파일명 (Drive면 경로 수정)
ROOT = Path("/content/lhk_cuda_bundle")
if not ROOT.exists():
    with zipfile.ZipFile(BUNDLE_ZIP) as z: z.extractall("/content")
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(GPU 런타임으로 변경 필요!)")
# data.yaml 절대경로 보정 (ultralytics용)
dy = (ROOT/"data.yaml").read_text()
if "path: ." in dy: (ROOT/"data.yaml").write_text(dy.replace("path: .", f"path: {ROOT}"))
print("bundle:", sorted(p.name for p in ROOT.iterdir()))

CUDA: True Tesla T4
bundle: ['.DS_Store', 'README.md', 'class_map.json', 'coco', 'cuda_baselines.json', 'data.yaml', 'images', 'labels', 'runs', 'val_gt.json']


In [11]:
# 공통 mAP@[0.75:0.95] 하니스 (MPS와 동일) — 예측 dts는 category_id=dl_idx, image_id=val 정렬순 1..51
import json, numpy as np
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
m2c = {int(k): v for k, v in json.load(open(ROOT/"class_map.json"))["model_index_to_category_id"].items()}
cocoGt = COCO(str(ROOT/"val_gt.json"))
val_imgs = sorted((ROOT/"images/val").glob("*.png"))
nid = {p.name: i+1 for i, p in enumerate(val_imgs)}
RESULTS = []
def score(name, dts, params_M):
    if not dts:
        mAP = ap75 = 0.0
    else:
        e = COCOeval(cocoGt, cocoGt.loadRes(dts), "bbox"); e.params.iouThrs = np.linspace(0.75, 0.95, 5)
        e.evaluate(); e.accumulate(); e.summarize(); mAP = float(e.stats[0])
        e2 = COCOeval(cocoGt, cocoGt.loadRes(dts), "bbox"); e2.params.iouThrs = np.array([0.75])
        e2.evaluate(); e2.accumulate(); e2.summarize(); ap75 = float(e2.stats[0])
    RESULTS.append({"model": name, "params_M": round(params_M, 1), "mAP_75_95": round(mAP, 4), "ap75": round(ap75, 4), "val_det": len(dts)})
    json.dump(RESULTS, open(ROOT/"cuda_baselines.json", "w"), ensure_ascii=False, indent=2)
    print(f">>> {name}: mAP@[0.75:0.95]={mAP:.4f}  AP@0.75={ap75:.4f}  val_det={len(dts)}")

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


## 1. RT-DETR (ultralytics) — 안정적, 먼저 실행

In [6]:
!pip -q install ultralytics
from ultralytics import RTDETR
m = RTDETR("rtdetr-l.pt")
m.train(data=str(ROOT/"data.yaml"), epochs=50, imgsz=640, batch=8, device=0, seed=42,
        project=str(ROOT/"runs"), name="rtdetr", exist_ok=True, plots=False, verbose=False)
best = RTDETR(str(ROOT/"runs/rtdetr/weights/best.pt"))
dts = []
for p in val_imgs:
    r = best.predict(str(p), conf=0.001, iou=0.6, max_det=20, device=0, verbose=False)[0]
    for b, c, s in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy(), r.boxes.conf.cpu().numpy()):
        x1, y1, x2, y2 = b
        dts.append({"image_id": nid[p.name], "category_id": m2c[int(c)], "bbox": [float(x1), float(y1), float(x2-x1), float(y2-y1)], "score": float(s)})
score("RT-DETR-l", dts, sum(pp.numel() for pp in best.model.parameters())/1e6)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 46.1 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.86 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/lhk_cuda_bundle/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/50      6.64G     0.5924      2.221      0.362         27        640: 100% ━━━━━━━━━━━━ 23/23 1.3s/it 29.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 2.0s/it 8.2s
                   all         51        166    0.00723     0.0844    0.00972    0.00836

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/50      7.04G     0.5484      2.397     0.3941         40        640: 0% ──────────── 0/23  0.7s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/50      7.08G     0.3497      2.601     0.2192         24        640: 100% ━━━━━━━━━━━━ 23/23 1.5it/s 15.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 3.0it/s 1.3s
                   all         51        166     0.0146      0.231    0.00964    0.00889

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       3/50      7.08G     0.2753      2.718     0.1305         49        640: 0% ──────────── 0/23  0.7s

...

      48/50      7.08G    0.06208     0.3204    0.03705         12        640: 100% ━━━━━━━━━━━━ 23/23 1.4it/s 17.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 3.0it/s 1.4s
                   all         51        166      0.893      0.713      0.777      0.762

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      49/50      7.08G    0.05442     0.2613    0.02894         25        640: 0% ──────────── 0/23  0.6s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      49/50      7.08G    0.06346     0.3253    0.03862         13        640: 100% ━━━━━━━━━━━━ 23/23 1.5it/s 15.8s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 2.8it/s 1.4s
                   all         51        166      0.892      0.716      0.778      0.765

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      50/50      7.08G    0.06128     0.3473     0.0424         27        640: 0% ──────────── 0/23  0.7s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      50/50      7.08G    0.06197     0.3222    0.03901         13        640: 100% ━━━━━━━━━━━━ 23/23 1.3it/s 17.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 3.0it/s 1.3s
                   all         51        166      0.889      0.717      0.778      0.765

50 epochs completed in 0.432 hours.
Optimizer stripped from /content/lhk_cuda_bundle/runs/rtdetr/weights/last.pt, 66.4MB
Optimizer stripped from /content/lhk_cuda_bundle/runs/rtdetr/weights/best.pt, 66.4MB

Validating /content/lhk_cuda_bundle/runs/rtdetr/weights/best.pt...
Ultralytics 8.4.86 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 310 layers, 32,098,820 parameters, 0 gradients, 103.7 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 2.6it/s 1.6s
                   all         51        166      0.898      0.688      0.786      0.767
Speed: 0.3ms prep

## 1b. Faster R-CNN (torchvision, warmup) — MPS서 RoIAlign CPU폴백으로 너무 느려(에폭 52분) CUDA로 이관
발산 방지: **선형 warmup + grad-clip**. (MPS 표의 0.0은 warmup 부재 발산 → 여기서 진짜 수치 산출)

In [7]:
import torch, time
from PIL import Image
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
Wp, Hp = 976, 1280

class DS(Dataset):
    def __init__(self, sp): self.im = sorted((ROOT/"images"/sp).glob("*.png")); self.sp = sp
    def __len__(self): return len(self.im)
    def __getitem__(self, i):
        p = self.im[i]; img = Image.open(p).convert("RGB"); bx = []; lb = []
        for ln in (ROOT/"labels"/self.sp/(p.stem+".txt")).read_text().splitlines():
            if not ln.strip(): continue
            m, cx, cy, nw, nh = ln.split(); m = int(m); cx, cy, nw, nh = map(float, (cx, cy, nw, nh))
            bx.append([(cx-nw/2)*Wp, (cy-nh/2)*Hp, (cx+nw/2)*Wp, (cy+nh/2)*Hp]); lb.append(m+1)
        return TF.to_tensor(img), {"boxes": torch.tensor(bx, dtype=torch.float32), "labels": torch.tensor(lb, dtype=torch.int64)}
def coll(b): return tuple(zip(*b))

mdl = fasterrcnn_resnet50_fpn(weights="DEFAULT")
mdl.roi_heads.box_predictor = FastRCNNPredictor(mdl.roi_heads.box_predictor.cls_score.in_features, 57)
mdl.transform.min_size = (512,); mdl.transform.max_size = 800; mdl.cuda()
dl = DataLoader(DS("train"), batch_size=4, shuffle=True, collate_fn=coll, num_workers=2)
opt = torch.optim.SGD([p for p in mdl.parameters() if p.requires_grad], lr=0.005, momentum=0.9, weight_decay=5e-4)
BL, EP = 0.005, 24; wu = min(500, len(dl)*2); g = 0; t0 = time.time()
for ep in range(EP):
    mdl.train(); tot = 0.0
    for im, tg in dl:
        g += 1
        if g <= wu:
            for pg in opt.param_groups: pg["lr"] = BL*(0.01+0.99*g/wu)
        im = [x.cuda() for x in im]; tg = [{k: v.cuda() for k, v in t.items()} for t in tg]
        loss = sum(mdl(im, tg).values())
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(mdl.parameters(), 10); opt.step(); tot += float(loss)
    if ep >= int(EP*0.8):
        for pg in opt.param_groups: pg["lr"] = BL*0.1
    print(f"ep{ep+1}/{EP} loss={tot/len(dl):.3f} ({time.time()-t0:.0f}s)")

mdl.eval(); dts = []
with torch.no_grad():
    for p in val_imgs:
        o = mdl([TF.to_tensor(Image.open(p).convert("RGB")).cuda()])[0]
        for b, l, sc in zip(o["boxes"].cpu().numpy(), o["labels"].cpu().numpy(), o["scores"].cpu().numpy()):
            if sc < 0.001: continue
            idx = int(l) - 1
            if idx not in m2c: continue
            x1, y1, x2, y2 = b
            dts.append({"image_id": nid[p.name], "category_id": m2c[idx], "bbox": [float(x1), float(y1), float(x2-x1), float(y2-y1)], "score": float(sc)})
score("FasterRCNN-R50(warmup)", dts, sum(q.numel() for q in mdl.parameters())/1e6)

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:00<00:00, 170MB/s]
/tmp/ipykernel_1950/2155450861.py:35: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(mdl.parameters(), 10); opt.step(); tot += float(loss)


ep1/24 loss=2.176 (23s)
ep2/24 loss=1.072 (46s)
ep3/24 loss=0.801 (69s)
ep4/24 loss=0.681 (93s)
ep5/24 loss=0.569 (117s)
ep6/24 loss=0.594 (141s)
ep7/24 loss=0.507 (166s)
ep8/24 loss=0.480 (191s)
ep9/24 loss=0.476 (215s)
ep10/24 loss=0.426 (240s)
ep11/24 loss=0.418 (265s)
ep12/24 loss=0.405 (290s)
ep13/24 loss=0.367 (315s)
ep14/24 loss=0.351 (340s)
ep15/24 loss=0.345 (365s)
ep16/24 loss=0.324 (389s)
ep17/24 loss=0.304 (415s)
ep18/24 loss=0.320 (440s)
ep19/24 loss=0.317 (464s)
ep20/24 loss=0.270 (489s)
ep21/24 loss=0.231 (514s)
ep22/24 loss=0.244 (538s)
ep23/24 loss=0.202 (562s)
ep24/24 loss=0.182 (586s)
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.07s).
Accumulating evaluation results...
DONE (t=0.07s).
 Average Precision  (AP) @[ IoU=0.75:0.95 | area=   all | maxDets=100 ] = 0.655
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = -1.000
 Average Precisi

## 2. mmdetection 설치 (Cascade · DINO 공용) — ⚠️ best-effort
Colab의 torch/CUDA에 맞춰 `mim`이 mmcv를 매칭. 실패 시 조정.

In [13]:
# mmdet 설치 (Colab Python 3.12 대응) — setuptools 먼저 고쳐 openmim/pkg_resources(ImpImporter) 에러 회피
import sys, torch; print("Python", sys.version.split()[0], "| torch", torch.__version__, "| cuda", torch.version.cuda)
!pip -q install -U pip "setuptools>=70" wheel
!pip -q install -U openmim
!mim install -q mmengine
!mim install -q "mmcv>=2.1.0,<2.3.0"
!mim install -q "mmdet>=3.3.0"
import mmengine, mmcv, mmdet
print("OK mmdet", mmdet.__version__, "| mmcv", mmcv.__version__)

Python 3.12.13 | torch 2.11.0+cu128 | cuda 12.8
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 60.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
openxlab 0.1.3 requires setuptools~=60.2.0, but you have setuptools 82.0.1 which is incompatible.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 82.0.1 which is incompatible.
pymc 5.28.5 requires rich>=13.7.1, but you have rich 13.4.2 which is incompatible.
pytensor 2.38.3 requires filelock>=3.15, but you have filelock 3.14.0 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 re

ModuleNotFoundError: No module named 'mmengine'

In [ ]:
# mmdet 학습 헬퍼: base config를 우리 fold0 COCO(56클래스)로 override → 학습 → val 추론 → 하니스 채점
from mmengine.config import Config
from mmengine.runner import Runner

CLASSES = tuple(str(m2c[i]) for i in range(56))
META = dict(classes=CLASSES)

def set_num_classes(model_cfg, nc=56):
    rh = model_cfg.get("roi_head")
    if rh is not None:                       # Cascade/Faster: roi_head.bbox_head (list 또는 dict)
        bh = rh["bbox_head"]
        for h in (bh if isinstance(bh, list) else [bh]): h["num_classes"] = nc
    if model_cfg.get("bbox_head") is not None:   # DINO/DETR 계열
        model_cfg["bbox_head"]["num_classes"] = nc

def run_mmdet(cfg_path, run_name, epochs=24, base_lr=None):
    cfg = Config.fromfile(cfg_path)
    for split, ann, pref in [("train", "coco/train.json", "images/train/"), ("val", "coco/val.json", "images/val/")]:
        dl = cfg[f"{split}_dataloader"]
        dl["dataset"].update(data_root=str(ROOT), ann_file=ann, data_prefix=dict(img=pref), metainfo=META)
    cfg.train_dataloader["batch_size"] = 2
    cfg.test_dataloader = cfg.val_dataloader
    cfg.val_evaluator.update(ann_file=str(ROOT/"coco/val.json"), metric="bbox"); cfg.test_evaluator = cfg.val_evaluator
    set_num_classes(cfg.model, 56)
    cfg.train_cfg["max_epochs"] = epochs
    if base_lr is not None: cfg.optim_wrapper["optimizer"]["lr"] = base_lr
    cfg.default_hooks["checkpoint"].update(interval=epochs, save_best="coco/bbox_mAP")
    cfg.work_dir = str(ROOT/f"runs/{run_name}")
    cfg.load_from = None  # 필요시 COCO 사전학습 ckpt URL 지정
    runner = Runner.from_cfg(cfg); runner.train()
    return runner, cfg

def mmdet_eval(run_name, cfg, label):
    from mmdet.apis import init_detector, inference_detector
    import glob
    ck = sorted(glob.glob(str(ROOT/f"runs/{run_name}/best_*.pth")) or glob.glob(str(ROOT/f"runs/{run_name}/epoch_*.pth")))[-1]
    model = init_detector(cfg, ck, device="cuda:0")
    dts = []
    for p in val_imgs:
        res = inference_detector(model, str(p)).pred_instances
        bb = res.bboxes.cpu().numpy(); lb = res.labels.cpu().numpy(); sc = res.scores.cpu().numpy()
        for (x1, y1, x2, y2), l, s in zip(bb, lb, sc):
            if s < 0.001: continue
            dts.append({"image_id": nid[p.name], "category_id": m2c[int(l)], "bbox": [float(x1), float(y1), float(x2-x1), float(y2-y1)], "score": float(s)})
    pm = sum(q.numel() for q in model.parameters())/1e6
    score(label, dts, pm)

## 3. Cascade R-CNN (고-IoU 정합 2-stage)

In [ ]:
r, c = run_mmdet("mmdet::cascade_rcnn/cascade-rcnn_r50_fpn_1x_coco.py", "cascade", epochs=24)
mmdet_eval("cascade", c, "Cascade-RCNN-R50")

## 4. DINO (정확도 트랜스포머) — 수렴 위해 에폭↑ 권장

In [ ]:
r, c = run_mmdet("mmdet::dino/dino-4scale_r50_8xb2-12e_coco.py", "dino", epochs=36)
mmdet_eval("dino", c, "DINO-4scale-R50")

## 5. DINOv2-frozen 백본 (선택·심화)
`torch.hub.load('facebookresearch/dinov2','dinov2_vits14')` 를 frozen 특징추출기로 쓰고 경량 검출 헤드를 얹는 방식. 배선이 커서 시간 여유 시 별도 진행(여기선 생략).

## 6. 결과 요약 · 내보내기

In [14]:
import pandas as pd
df = pd.DataFrame(RESULTS).sort_values("mAP_75_95", ascending=False)
print(df.to_string(index=False))
print("\n→ /content/lhk_cuda_bundle/cuda_baselines.json")
from google.colab import files  # Colab 전용
files.download(str(ROOT/"cuda_baselines.json"))

KeyError: 'mAP_75_95'